In [0]:
# Read the CSV from the volume
file_path = "/Volumes/airnz_catalog/airnzschema/airnzvolume/AirNZ_Flights_500K.csv"

df_txt = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .csv(file_path))

print(f"Total rows: {df_txt.count()}")
display(df_txt.limit(5))

Total rows: 500000


FLIGHT_DATE,FLIGHT_TIME,TIME_OF_DAY,AIRLINE_CD,FLIGHT_NO,DEPARTURE_STATION_CD,ARRIVAL_STATION_CD,ARRIVAL_COUNTRY,ARRIVAL_REGION,HAUL,AIRCRAFT_TYPE,FIRST_CLASS_SEATS,BUSINESS_PREMIER_SEATS,PREMIUM_ECONOMY_SEATS,ECONOMY_SEATS,TIER1_ELIGIBLE_PAX,TIER2_ELIGIBLE_PAX,TIER3_ELIGIBLE_PAX,YEAR,MONTH,MONTH_NAME,WEEK,DAY,DAY_OF_WEEK,DAY_NAME,HOUR,QUARTER,IS_WEEKEND,SEASON,DEPARTURE_DELAY_MIN,DELAY_CATEGORY
2025-09-02,14:19:00,Afternoon,NZ,NZ5211,AKL,HND,Japan,Asia,LONG,B777-300ER,0,49,39,178,2,13,26,2025,9,Sep,36,2,1,Tue,14,3,0,Spring,15,Minor
2025-06-10,06:42:00,Morning,NZ,NZ7282,AKL,HND,Japan,Asia,LONG,B777-300ER,0,49,28,178,4,18,41,2025,6,Jun,24,10,1,Tue,6,2,0,Winter,20,Moderate
2025-10-27,15:33:00,Afternoon,NZ,NZ1896,AKL,TBU,Tonga,Pacific Islands,SHORT,A320neo,0,18,0,168,0,0,65,2025,10,Oct,44,27,0,Mon,15,4,0,Spring,5,Minor
2025-06-15,18:29:00,Evening,NZ,NZ5497,AKL,HIR,Solomon Islands,Pacific Islands,SHORT,A321neo,0,18,0,172,3,0,25,2025,6,Jun,24,15,6,Sun,18,2,1,Winter,4,Minor
2025-08-25,20:35:00,Evening,NZ,NZ1493,AKL,TBU,Tonga,Pacific Islands,SHORT,A320neo,0,18,0,168,3,0,22,2025,8,Aug,35,25,0,Mon,20,3,0,Winter,2,Minor


In [0]:
from pyspark.sql.functions import to_date, col

# Clean and cast data
clean_df = (df_txt
    .withColumn("FLIGHT_DATE", to_date(col("FLIGHT_DATE"), "yyyy-MM-dd"))
    .dropDuplicates(["FLIGHT_DATE", "FLIGHT_TIME", "FLIGHT_NO", "ARRIVAL_STATION_CD"])
    .na.drop()
)

# Save as Delta Table
(clean_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("airnz_catalog.airnzschema.airnz_flights_clean"))

In [0]:
from pyspark.sql.functions import count, avg

daily_stats = (clean_df
    .groupBy("FLIGHT_DATE")
    .agg(
        count("FLIGHT_NO").alias("num_flights"),
        avg("DEPARTURE_DELAY_MIN").alias("avg_delay")
    )
    .orderBy("FLIGHT_DATE"))

display(daily_stats)

FLIGHT_DATE,num_flights,avg_delay
2025-01-01,1337,14.835452505609574
2025-01-02,1365,14.252014652014653
2025-01-03,1322,15.097579425113464
2025-01-04,1264,15.01740506329114
2025-01-05,1259,14.596505162827642
2025-01-06,1375,14.416
2025-01-07,1329,14.92249811888638
2025-01-08,1385,14.653429602888087
2025-01-09,1331,14.274981217129977
2025-01-10,1334,14.715892053973013


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col, lit, sum as _sum, when, round as _round
# ==========================================
# 1) MAP STEP
# ==========================================
# Process each row independently and emit a key-value pair
mapped_df = (clean_df
    .select(
        col("FLIGHT_DATE").alias("date"),
        col("DEPARTURE_DELAY_MIN").alias("delay"),
        lit(1).alias("count")
    ))

print("--- MAP OUTPUT ---")
mapped_df.show(10, truncate=False)

--- MAP OUTPUT ---
+----------+-----+-----+
|date      |delay|count|
+----------+-----+-----+
|2025-01-01|16   |1    |
|2025-01-01|23   |1    |
|2025-01-01|28   |1    |
|2025-01-01|15   |1    |
|2025-01-01|23   |1    |
|2025-01-01|3    |1    |
|2025-01-01|1    |1    |
|2025-01-01|6    |1    |
|2025-01-01|23   |1    |
|2025-01-01|6    |1    |
+----------+-----+-----+
only showing top 10 rows


In [0]:
# ==========================================
# 2) REDUCE STEP
# ==========================================
# Group by the key (date) and aggregate the sums
reduced_df = (mapped_df
    .groupBy("date")
    .agg(
        _sum("delay").alias("total_delay"),
        _sum("count").alias("total_count")
    ))

print("--- REDUCE OUTPUT ---")
reduced_df.show(10, truncate=False)

--- REDUCE OUTPUT ---
+----------+-----------+-----------+
|date      |total_delay|total_count|
+----------+-----------+-----------+
|2025-04-17|18568      |1283       |
|2025-07-11|23851      |1372       |
|2025-02-05|18695      |1277       |
|2025-04-07|19382      |1380       |
|2025-09-15|20015      |1353       |
|2025-03-10|18875      |1322       |
|2025-11-01|19362      |1325       |
|2025-11-25|19130      |1321       |
|2025-08-13|23310      |1356       |
|2025-09-22|19307      |1352       |
+----------+-----------+-----------+
only showing top 10 rows


In [0]:
# ==========================================
# 3) POST-PROCESSING (Average)
# ==========================================
# Compute the final daily average
daily_avg = (reduced_df
    .withColumn("daily_avg_delay",
        when(col("total_count") > 0, col("total_delay") / col("total_count")).otherwise(None))
    .withColumn("daily_avg_delay", _round("daily_avg_delay", 2))
    .select("date", "daily_avg_delay")
    .orderBy("date"))

display(daily_avg)

date,daily_avg_delay
2025-01-01,14.84
2025-01-02,14.25
2025-01-03,15.1
2025-01-04,15.02
2025-01-05,14.6
2025-01-06,14.42
2025-01-07,14.92
2025-01-08,14.65
2025-01-09,14.27
2025-01-10,14.72


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, dayofweek, month, when, col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import RegressionEvaluator, BinaryClassificationEvaluator

print("1. Preparing Time-Series Features...")
# Create lag features (previous days' delays)
w = Window.orderBy("FLIGHT_DATE")
ml_data = (daily_stats
    .withColumn("lag_1_delay", lag("avg_delay").over(w))
    .withColumn("lag_7_delay", lag("avg_delay", 7).over(w))
    .withColumn("lag_1_flights", lag("num_flights").over(w))
    .withColumn("day_of_week", dayofweek("FLIGHT_DATE"))
    .withColumn("month", month("FLIGHT_DATE"))
    .na.drop() # Drop the first 7 days which have null values due to lagging
)

# Combine features into a single vector for ML
feature_cols = ["lag_1_delay", "lag_7_delay", "lag_1_flights", "day_of_week", "month"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
ml_df = assembler.transform(ml_data)

1. Preparing Time-Series Features...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# ---------------------------------------------------------
print("2. Training Regression Model (Predicting exact delay)...")
# Split data 80/20
train_reg, test_reg = ml_df.randomSplit([0.8, 0.2], seed=42)

# Train Random Forest Regressor
rf_reg = RandomForestRegressor(featuresCol="features", labelCol="avg_delay", numTrees=50)
reg_model = rf_reg.fit(train_reg)

# Make predictions and evaluate
reg_preds = reg_model.transform(test_reg)
rmse = RegressionEvaluator(labelCol="avg_delay", metricName="rmse").evaluate(reg_preds)
r2 = RegressionEvaluator(labelCol="avg_delay", metricName="r2").evaluate(reg_preds)

print(f"--> Regression RMSE: {rmse:.2f}")
print(f"--> Regression R2 Score: {r2:.2f}")

2. Training Regression Model (Predicting exact delay)...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


--> Regression RMSE: 0.35
--> Regression R2 Score: 0.92


In [0]:
# ---------------------------------------------------------
print("3. Training Classification Model (Predicting if delay > 15 mins)...")
# Create binary label: 1 if delay > 15, else 0
cls_df = ml_df.withColumn("label", when(col("avg_delay") > 15, 1.0).otherwise(0.0))
train_cls, test_cls = cls_df.randomSplit([0.8, 0.2], seed=42)

# Train Random Forest Classifier
rf_cls = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50)
cls_model = rf_cls.fit(train_cls)

# Make predictions and evaluate
cls_preds = cls_model.transform(test_cls)
auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC").evaluate(cls_preds)

print(f"--> Classification AUC Score: {auc:.2f}")

3. Training Classification Model (Predicting if delay > 15 mins)...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


--> Classification AUC Score: 0.89


In [0]:
# ---------------------------------------------------------
print("4. Displaying Results Tables...")
print("Regression Predictions (Actual vs Predicted Delay):")
display(reg_preds.select("FLIGHT_DATE", "avg_delay", "prediction").limit(5))

print("Classification Predictions (Actual vs Predicted Class):")
display(cls_preds.select("FLIGHT_DATE", "avg_delay", "label", "prediction").limit(5))

4. Displaying Results Tables...
Regression Predictions (Actual vs Predicted Delay):


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


FLIGHT_DATE,avg_delay,prediction
2025-01-10,14.715892053973013,14.690694593798007
2025-01-14,14.576615831517792,14.696275604142635
2025-01-16,14.513287775246773,14.684182680807872
2025-01-21,14.71356783919598,14.661484181686504
2025-01-27,14.534558823529412,14.681163676134693


Classification Predictions (Actual vs Predicted Class):


FLIGHT_DATE,avg_delay,label,prediction
2025-01-10,14.715892053973013,0.0,0.0
2025-01-14,14.576615831517792,0.0,0.0
2025-01-16,14.513287775246773,0.0,0.0
2025-01-21,14.71356783919598,0.0,0.0
2025-01-27,14.534558823529412,0.0,0.0
